In [4]:
import pandas as pd

# Load dataset (adjust path if needed)
df = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv copy.gz", low_memory=False)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (2260701, 151)

Columns: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import pandas as pd
import numpy as np

# 1) Keep only columns we actually need (risk + business)
cols = [
    "loan_amnt","term","int_rate","installment",
    "grade","sub_grade",
    "emp_length","home_ownership","annual_inc","verification_status",
    "purpose","addr_state",
    "dti","delinq_2yrs","fico_range_low","fico_range_high",
    "inq_last_6mths","open_acc","pub_rec","revol_bal","revol_util","total_acc",
    "application_type","issue_d","earliest_cr_line",
    "loan_status"
]
df2 = df[cols].copy()

# 2) Keep only statuses needed for a clean binary default target
df2 = df2[df2["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
df2["default"] = (df2["loan_status"] == "Charged Off").astype(int)

# Quick KPI
default_rate = df2["default"].mean()
print("Rows after filtering:", df2.shape)
print("Default rate:", round(default_rate*100, 2), "%")
df2["loan_status"].value_counts()

Rows after filtering: (1345310, 27)
Default rate: 19.96 %


loan_status
Fully Paid     1076751
Charged Off     268559
Name: count, dtype: int64

In [6]:
grade_default = (
    df2.groupby("grade")["default"]
    .mean()
    .sort_index()
    .reset_index()
)

grade_default["default_rate_%"] = grade_default["default"] * 100
grade_default

,grade,default,default_rate_%
0,A,0.060407,6.040665
1,B,0.133852,13.385157
2,C,0.224396,22.439649
3,D,0.303822,30.382229
4,E,0.384784,38.478377
5,F,0.452024,45.202446
6,G,0.499343,49.934297


In [9]:
exposure_by_grade = (
    df2.groupby("grade")
    .agg(
        total_loans=("loan_amnt", "count"),
        total_exposure=("loan_amnt", "sum"),
        default_rate=("default", "mean")
    )
    .reset_index()
)

exposure_by_grade["expected_loss"] = (
    exposure_by_grade["total_exposure"] *
    exposure_by_grade["default_rate"]
)

exposure_by_grade

,grade,total_loans,total_exposure,default_rate,expected_loss
0,A,235090,3.265953e+09,0.060407,1.972853e+08
1,B,392741,5.198962e+09,0.133852,6.958892e+08
2,C,381686,5.415526e+09,0.224396,1.215225e+09
3,D,200953,3.069036e+09,0.303822,9.324415e+08
4,E,93650,1.649930e+09,0.384784,6.348663e+08
5,F,32058,6.119090e+08,0.452024,2.765978e+08
6,G,9132,1.880170e+08,0.499343,9.388495e+07


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Select numeric modeling features
model_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_mid",
    "revol_util",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "income_to_loan",
    "loan_to_income"
]

df_model = df2[model_cols + ["default"]].dropna()

X = df_model[model_cols]
y = df_model["default"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

KeyError: "['fico_mid', 'income_to_loan', 'loan_to_income'] not in index"

In [11]:
import numpy as np
import pandas as pd

# If you haven't created fico_mid yet:
if "fico_mid" not in df2.columns:
    # some datasets use fico_range_low/high (yours has both)
    df2["fico_mid"] = (df2["fico_range_low"] + df2["fico_range_high"]) / 2

# If you haven't created income ratios yet:
if "income_to_loan" not in df2.columns:
    df2["income_to_loan"] = df2["annual_inc"] / df2["loan_amnt"]

if "loan_to_income" not in df2.columns:
    df2["loan_to_income"] = df2["loan_amnt"] / df2["annual_inc"]

# Quick check
[c for c in ["fico_mid","income_to_loan","loan_to_income"] if c in df2.columns]


['fico_mid', 'income_to_loan', 'loan_to_income']

In [12]:
# Ensure numeric fields are numeric
df2["revol_util"] = pd.to_numeric(df2["revol_util"], errors="coerce")
df2["int_rate"] = pd.to_numeric(df2["int_rate"], errors="coerce")
df2["term"] = pd.to_numeric(df2["term"], errors="coerce")
df2["annual_inc"] = pd.to_numeric(df2["annual_inc"], errors="coerce")
df2["dti"] = pd.to_numeric(df2["dti"], errors="coerce")
df2["loan_amnt"] = pd.to_numeric(df2["loan_amnt"], errors="coerce")
df2["installment"] = pd.to_numeric(df2["installment"], errors="coerce")

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

model_cols = [
    "loan_amnt","term","int_rate","installment",
    "annual_inc","dti","fico_mid","revol_util",
    "delinq_2yrs","inq_last_6mths","open_acc","total_acc",
    "income_to_loan","loan_to_income"
]

df_model = df2[model_cols + ["default"]].replace([np.inf, -np.inf], np.nan).dropna()

X = df_model[model_cols]
y = df_model["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000, n_jobs=-1)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Model rows used:", df_model.shape)
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

ValueError: With n_samples=0, test_size=0.3 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [14]:
import numpy as np
import pandas as pd

model_cols = [
    "loan_amnt","term","int_rate","installment",
    "annual_inc","dti","fico_mid","revol_util",
    "delinq_2yrs","inq_last_6mths","open_acc","total_acc",
    "income_to_loan","loan_to_income"
]

# How many non-null values each column has
nn = df2[model_cols + ["default"]].notna().sum().sort_values()
nn

term                    0
revol_util        1344453
dti               1344936
inq_last_6mths    1345309
loan_amnt         1345310
int_rate          1345310
installment       1345310
annual_inc        1345310
fico_mid          1345310
delinq_2yrs       1345310
open_acc          1345310
total_acc         1345310
income_to_loan    1345310
loan_to_income    1345310
default           1345310
dtype: int64

In [15]:
df2[["term","int_rate","revol_util"]].head(10)

,term,int_rate,revol_util
0,NaN,13.99,29.7
1,NaN,11.99,19.2
2,NaN,10.78,56.2
4,NaN,22.45,64.5
5,NaN,13.44,68.4
6,NaN,9.17,84.5
7,NaN,8.49,5.7
8,NaN,6.49,34.5
9,NaN,11.48,39.1
12,NaN,12.88,67.2


In [16]:
# Clean term column correctly
df2["term"] = df2["term"].astype(str).str.strip()
df2["term"] = df2["term"].str.extract(r"(\d+)", expand=False)
df2["term"] = pd.to_numeric(df2["term"], errors="coerce")

df2["term"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: term, dtype: float64

In [17]:
df2["term"].value_counts()

Series([], Name: count, dtype: int64)

In [18]:
import pandas as pd

# Restore raw term values from the original dataframe using the same row index
df2["term_raw"] = df.loc[df2.index, "term"].astype(str)

# Parse to numeric months (36 / 60)
df2["term"] = (
    df2["term_raw"]
    .str.replace("\u00A0", " ", regex=False)  # handles non-breaking spaces
    .str.strip()
    .str.extract(r"(\d+)", expand=False)
)

df2["term"] = pd.to_numeric(df2["term"], errors="coerce")

print(df2["term"].isna().mean(), "fraction NaN in term")
print(df2["term"].value_counts(dropna=False).head(10))
df2[["term_raw","term"]].head(10)

0.0 fraction NaN in term
term
36    1020743
60     324567
Name: count, dtype: int64


,term_raw,term
0,36 months,36
1,36 months,36
2,60 months,60
4,60 months,60
5,36 months,36
6,36 months,36
7,36 months,36
8,36 months,36
9,36 months,36
12,36 months,36


In [19]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

model_cols = [
    "loan_amnt","term","int_rate","installment",
    "annual_inc","dti","fico_mid","revol_util",
    "delinq_2yrs","inq_last_6mths","open_acc","total_acc",
    "income_to_loan","loan_to_income"
]

df_model = df2[model_cols + ["default"]].replace([np.inf, -np.inf], np.nan).copy()

# Fill missing with median (baseline)
for c in model_cols:
    df_model[c] = pd.to_numeric(df_model[c], errors="coerce")
    df_model[c] = df_model[c].fillna(df_model[c].median())

print("df_model shape:", df_model.shape)
print("Any NaNs left?:", df_model.isna().sum().sum())

X = df_model[model_cols]
y = df_model["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000, n_jobs=-1)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

df_model shape: (1345310, 15)
Any NaNs left?: 0


/Users/saikumarsheri/Desktop/projects/loan-default-risk-analysis/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


ROC-AUC: 0.7026330117538435

Confusion Matrix:
 [[317698   5327]
 [ 75197   5371]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.98      0.89    323025
           1       0.50      0.07      0.12     80568

    accuracy                           0.80    403593
   macro avg       0.66      0.53      0.50    403593
weighted avg       0.75      0.80      0.73    403593



In [20]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

# Try different probability thresholds
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    y_pred_thresh = (y_proba >= t).astype(int)
    precision = precision_score(y_test, y_pred_thresh)
    recall = recall_score(y_test, y_pred_thresh)
    print(f"Threshold {t}: Precision={precision:.3f}, Recall={recall:.3f}")

Threshold 0.5: Precision=0.502, Recall=0.067
Threshold 0.4: Precision=0.460, Recall=0.164
Threshold 0.3: Precision=0.400, Recall=0.340
Threshold 0.2: Precision=0.318, Recall=0.631
Threshold 0.1: Precision=0.231, Recall=0.945


In [21]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": model_cols,
    "coefficient": model.coef_[0]
})

feature_importance["abs_coef"] = feature_importance["coefficient"].abs()
feature_importance.sort_values("abs_coef", ascending=False)

,feature,coefficient,abs_coef
2,int_rate,0.414761,0.414761
6,fico_mid,-0.240197,0.240197
1,term,0.221935,0.221935
12,income_to_loan,-0.159648,0.159648
11,total_acc,-0.159616,0.159616
5,dti,0.140315,0.140315
10,open_acc,0.124805,0.124805
4,annual_inc,-0.113016,0.113016
9,inq_last_6mths,0.063541,0.063541
7,revol_util,-0.056676,0.056676


In [22]:
# Create full prediction dataset
df_scoring = df_model.copy()

# Predict probability for entire dataset
X_scaled_full = scaler.transform(df_model[model_cols])
df_scoring["default_probability"] = model.predict_proba(X_scaled_full)[:, 1]

# Convert to 0-100 risk score
df_scoring["risk_score"] = (df_scoring["default_probability"] * 100).round(2)

df_scoring[["default_probability","risk_score"]].describe()

,default_probability,risk_score
count,1.345310e+06,1.345310e+06
mean,1.996169e-01,1.996169e+01
std,1.169443e-01,1.169443e+01
min,1.275831e-40,0.000000e+00
25%,1.155631e-01,1.156000e+01
50%,1.720790e-01,1.721000e+01
75%,2.544139e-01,2.544000e+01
max,9.999984e-01,1.000000e+02


In [25]:
df_export = df2.copy()

# Merge risk score back to main dataset
df_export = df_export.loc[df_model.index]
df_export["risk_score"] = df_scoring["risk_score"]

df_export.to_csv("../data/processed/loan_risk_scored_dataset.csv", index=False)

print("Export complete.")

OSError: Cannot save file into a non-existent directory: '../data/processed'

In [26]:
import os

# Create processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

df_export.to_csv("../data/processed/loan_risk_scored_dataset.csv", index=False)

print("Export complete.")

Export complete.
